# Can we do kingdoms

In [ ]:
#| default_exp game/property

## The politcs of Geography
This is part of a larger game design project where I create a board game generator for my son. I created code that can load or maps with different elevations. from there we can run weather and climate simulators (as well as some terraforming to make sure that rivers wind up going somewhere). The next step is to try to create countries.

At some level, the mistakes are more interesting than the results. Because I use water as what matters, you can see why Sacramento became the capital of California. By no means does this have any historical or real accuracy. but I do think life is better with cool pictures

In [ ]:
#| export
import sys
import math
import numpy as np
import pandas as pd
import random
from fastcore.basics import patch
import heapq # for shortest path
from importlib import resources

In [ ]:
from dataclasses import dataclass, field 
from typing import Optional, List, Set , Callable
from enum import Enum 
import uuid

In [ ]:
#| export
#This is kitchen sink approach to the library. I just didn't know what I needed

from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover


In [ ]:
#| export
from HexMagic.game.settlement import CountryFlag

In [ ]:
#| export
from HexMagic.weather import TerraDemo
from HexMagic.geology import  SoilSystem, DrainageBasins, Geology

In [ ]:
class Resources(Enum):
    """Available Foods for pieces"""
    GRAIN = "GRAIN"
    CHICKEN = "CHICKEN"
    VEGATABLES = "VEGATABLES"
    FRUIT = "FRUIT"
    FISH = "FISH"
    HERD = "HERD"
    

In [ ]:
@dataclass 
class Piece: 
    """A game piece representing a group of units."""

    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0  # Kingdom/country ID
    parent_id: Optional[str] = None  # ID of piece that spawned this
    location:int = None

    # Core attributes
    size: int = 100  # Number of units in this piece
    health: int = 100  # Hit points (0-100 scale)
    max_health: int = 100

    # Vision and intelligence
    sight: int = 3  # How many hex rings they can see
    memory: float = 0.8  # Retention rate (0-1, higher = better memory)

    # Movement
    movement_range: int = 4  # Max weighted hexes per turn
    current_position: int = -1  # Current hex index


    # Settlement state
    harvest_goal: Resources = None
    settle_progress: int = 0  # Turns spent settling (settlement complete at threshold)
    settle_threshold: int = 3
    db: GeoStorage = None # this should not be encoded or decoded

    PIECE_FIELDS = ['id','owner_id','parent_id','location','size','health',
                'max_health','sight','memory','movement_range',
                'current_position','harvest_goal','settle_progress','settle_threshold']

    def encode(self) -> str:
        """Encode piece as a single tab-delimited line."""
        vals = []
        for f in self.PIECE_FIELDS:
            v = getattr(self, f)
            if v is None:        vals.append('')
            elif isinstance(v, Resources): vals.append(v.value)
            else:                vals.append(str(v))
        return '\t'.join(vals)

    @staticmethod
    def decode(line: str) -> 'Piece':
        """Decode a tab-delimited line into a Piece."""
        parts = line.split('\t')
        kw = dict(zip(PIECE_FIELDS, parts))
        # Convert types back
        for f in ['owner_id','size','health','max_health','sight',
                'movement_range','current_position','settle_progress','settle_threshold']:
            kw[f] = int(kw[f]) if kw[f] else 0
        for f in ['location']:
            kw[f] = int(kw[f]) if kw[f] else None
        kw['memory'] = float(kw['memory']) if kw['memory'] else 0.8
        kw['parent_id'] = kw['parent_id'] or None
        kw['harvest_goal'] = Resources(kw['harvest_goal']) if kw['harvest_goal'] else None
        return Piece(**kw)



## Settlement

In [ ]:
@dataclass 
class Settlement: 
    """A game piece representing a group of units."""

    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0
    location: int = None
    name:str = ""
    # Core attributes
    size: int = 100
    health: int = 100
    citizens: List[Piece] = field(default_factory=list)

    SETTLEMENT_FIELDS = ['id', 'owner_id', 'name', 'location', 'size', 'health']

    def encode(self) -> str:
        """Encode settlement as tab-delimited lines. First line is settlement fields, remaining lines are citizen pieces."""
        vals = []
        for f in self.SETTLEMENT_FIELDS:
            v = getattr(self, f)
            vals.append('' if v is None else str(v))
        lines = ['\t'.join(vals)]
        for c in self.citizens:
            lines.append(c.encode())
        return '\n'.join(lines)

    @staticmethod
    def decode(text: str) -> 'Settlement':
        """Decode a settlement from encoded text. First line is settlement, remaining lines are citizen pieces."""
        lines = text.strip().split('\n')
        parts = lines[0].split('\t')
        kw = dict(zip(Settlement.SETTLEMENT_FIELDS, parts))
        for f in ['owner_id', 'size', 'health']:
            kw[f] = int(kw[f]) if kw[f] else 0
        kw['location'] = int(kw['location']) if kw['location'] else None
        citizens = [Piece.decode(l) for l in lines[1:] if l.strip()]
        return Settlement(**kw, citizens=citizens)


I need an encode and decode for Settlement. Can this be a dataclass

In [ ]:
#| export
from typing import NamedTuple

class WatershedFlow(NamedTuple):
    """ a place holder so we don't recompute flows which can be expensive """
    watershed: Watershed
    flow: float


   

In [ ]:
#| export
class TradeRoute:
    """ going to figure how to link up what we build"""

    def __init__(self, path: [HexPosition],  origin = 0, cost: float=0,  name: str=""):
        self.path = path
        self.cost = cost
        self.name = name
        self.origin = origin

    def destination(self)->HexPosition:
        return self.path[-1]

    @staticmethod
    def decode(s: str) -> 'TradeRoute':
        """Decode TradeRoute from string format."""
        name = ""
        origin = 0
        cost = 0.0
        path = []
        
        for line in s.strip().split('\n'):
            if ':' not in line:
                continue
            key, val = line.split(':', 1)
            if key == 'name':
                name = val
            elif key == 'origin':
                origin = int(val)
            elif key == 'cost':
                cost = float(val)
            elif key == 'path':
                path = [int(h) for h in val.split(',') if h]
        
        return TradeRoute(path=path, origin=origin, cost=cost, name=name)


In [ ]:
#| export
@patch
def encode(self: TradeRoute) -> str:
    """Encode TradeRoute to string format."""
    path_str = ','.join(str(h) for h in self.path)
    lines = [
        f"name:{self.name}",
        f"origin:{self.origin}",
        f"cost:{self.cost}",
        f"path:{path_str}"
    ]
    return '\n'.join(lines)

In [ ]:
#| export
@patch
def encode(self: HexRegion) -> str:
    """Encode HexRegion as comma-separated hex indices."""
    return ','.join(str(h) for h in sorted(self.hexes))

@staticmethod
def _decode_hexregion(s: str, hexGrid: HexGrid) -> HexRegion:
    """Decode HexRegion from comma-separated hex indices."""
    hexes = set(int(h) for h in s.split(',') if h)
    return HexRegion(hexes=hexes, hexGrid=hexGrid)

HexRegion.decode = _decode_hexregion


## Kingdom

In [ ]:
#| export
class Kingdom:
    """ This is a terrority run by a single government"""
    def __init__(self, capital_hex: int, region: HexRegion, world: Geology,countryId:int = 0, flag = None):
        capital = Settlement(owner_id=countryId,location=capital_hex)
        self.settlements = [capital]  # capital is first
        self.region = region
        self.world = world
        self.countryId = countryId
        self.captured:[WatershedFlow] = []
        self.routes:[TradeRoute] = []
        self.flag = flag
        self.countryName = ""

        
    
    @classmethod
    def start(cls, world: Geology, scored:[WatershedFlow],palette_name: str = "husl"):
        terrain = world.terrain
        """Create initial kingdoms from best watersheds."""
        kingdoms = []
        i = 1 # lets reserve 0 for unclaimed and -1 for unavailable

    
        # Generate styles using seaborn palette
        flags = CountryFlag.seaborn(palette_name, levels=len(scored)+2) # just so we can index by 1
        saturation = 0.7

        for score in scored:
            ws = score.watershed
            capital = ws.max_flow_hex()[0]
          
            kingdom = cls(capital, ws.region, world, countryId=i,flag=flags[i])
            i += 1
            kingdom.captured.append(score)
            kingdoms.append(kingdom)
        
        return kingdoms

    @staticmethod
    def decode(s: str, world: Geology) -> 'Kingdom':
        """Decode Kingdom from string format."""
        lines = s.strip().split('\n')
        
        countryId = 0
        countryName = ""
        settlements = []
        flag = None
        region = None
        captured = []
        routes = []
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('countryId:'):
                countryId = int(line.split(':', 1)[1])
            elif line.startswith('countryName:'):
                countryName = line.split(':', 1)[1]
            elif line.startswith('flag:'):
                flag = CountryFlag.decode(line.split(':', 1)[1])
            elif line.startswith('+settlements'):
                i += 1
                while i < len(lines) and not lines[i].startswith('-settlements'):
                    if lines[i].startswith('+settlement'):
                        settle_lines = []
                        i += 1
                        while i < len(lines) and not lines[i].startswith('-settlement'):
                            settle_lines.append(lines[i])
                            i += 1
                        settlements.append(Settlement.decode('\n'.join(settle_lines)))
                    i += 1
            elif line.startswith('+region'):
                region_lines = []
                i += 1
                while i < len(lines) and not lines[i].startswith('-region'):
                    region_lines.append(lines[i])
                    i += 1
                region = HexRegion.decode('\n'.join(region_lines), world.terrain.hexGrid)
            elif line.startswith('+captured:'):
                i += 1
                while i < len(lines) and not lines[i].startswith('-captured'):
                    if lines[i].startswith('+watershed:'):
                        flow = float(lines[i].split(':', 1)[1])
                        ws_lines = []
                        i += 1
                        while i < len(lines) and not lines[i].startswith('-watershed'):
                            ws_lines.append(lines[i])
                            i += 1
                        ws = Watershed.decode('\n'.join(ws_lines), world.terrain)
                        captured.append(WatershedFlow(ws, flow))
                    i += 1
            elif line.startswith('+routes:'):
                i += 1
                route_lines = []
                while i < len(lines) and not lines[i].startswith('-routes'):
                    if lines[i] == '---':
                        if route_lines:
                            routes.append(TradeRoute.decode('\n'.join(route_lines)))
                        route_lines = []
                    else:
                        route_lines.append(lines[i])
                    i += 1
            i += 1
        
        # Build kingdom using first settlement's location as capital_hex
        capital_hex = settlements[0].location if settlements else 0
        kingdom = Kingdom(capital_hex, region, world, countryId, flag)
        kingdom.settlements = settlements
        kingdom.countryName = countryName
        kingdom.captured = captured
        kingdom.routes = routes
        
        return kingdom





In [ ]:
#| export
@patch
def encode(self: Kingdom) -> str:
    """Encode Kingdom to string format."""
    lines = [
        f"countryId:{self.countryId}",
        f"countryName:{self.countryName}",
    ]
    
    # Flag
    if self.flag:
        lines.append(f"flag:{self.flag.encode()}")
    
    # Settlements
    if self.settlements:
        lines.append(f"+settlements:{len(self.settlements)}")
        for s in self.settlements:
            lines.append("+settlement")
            lines.append(s.encode())
            lines.append("-settlement")
        lines.append("-settlements")
    
    # Region
    lines.append("+region")
    lines.append(self.region.encode())
    lines.append("-region")
    
    # Captured watersheds
    if self.captured:
        lines.append(f"+captured:{len(self.captured)}")
        for wf in self.captured:
            lines.append(f"+watershed:{wf.flow}")
            lines.append(wf.watershed.encode())
            lines.append("-watershed")
        lines.append("-captured")
    
    # Routes
    if self.routes:
        lines.append(f"+routes:{len(self.routes)}")
        for route in self.routes:
            lines.append(route.encode())
            lines.append("---")
        lines.append("-routes")
    
    return '\n'.join(lines)


I need to modify Kingdom encode and decode so it handles settlements

please write as you suggested

## Details

In [ ]:
class CountryDetails:
    def __init__(self,country:Kingdom):
        self.country = country
        self.updateParent(country.world.terrain)

    def updateParent(self,parent:Terrain):
        self.parent = parent
        grid, subregion, regionMapper = self.country.region.crop_to_centered_grid(style=StyleCSS("base",fill="lightgray",stroke="blue"))
        self.subregion = subregion
        self.regionMapper = regionMapper
        self.countryMap = Terrain(bounds = grid.bounds,
            radius = grid.radius,
            colorLevels= parent.colorLevels,
            seaLevel = parent.seaLevel,
            elevationDelta = parent.elevationDelta,
            geo = parent.geo,
            climate = parent.climate
        )
        numHexes = len(grid.hexes)
        self.countryMap.hexGrid = grid
        self.countryMap.elevations = np.zeros(numHexes)

        field_names = list(parent.fields.keys())
        
        for aField in field_names:
            self.countryMap.fields[aField] = np.zeros(numHexes)

        mapper = {}
        for dest in range(numHexes):
            source = regionMapper(dest)
            self.countryMap.elevations[dest] = parent.elevations[source]
            mapper[source] = dest
            for aField in field_names:
                self.countryMap.fields[aField][dest] = parent.fields[aField][source]
        self.mapper = mapper
       

In [ ]:


@patch
def viewableRegion(self:CountryDetails,region)->HexRegion:
    """ return a region that have hexes that work """
    hexes = set([self.mapper[i] for i in region.hexes if i in self.mapper])
    return HexRegion(hexes=hexes, hexGrid = self.countryMap.hexGrid)



In [ ]:
@patch
def clearUnknowns(self:CountryDetails):
    clearC = StyleCSS("blank",fill="none",stroke="none")
    self.countryMap.hexGrid.builder.add_style(clearC)
    for i in range(len(self.countryMap.hexGrid.hexes)):
        if self.regionMapper(i) < 0:
            self.countryMap.hexGrid.hexes[i].style = clearC

## Board

In [ ]:
#| export
class GameBoard:
    """ This lets us keep track of many of the games global properties"""

    def __init__(self,terrain,top_n=3,year=1900,gender=None):
        self.terrain = terrain
        self.world = Geology(terrain,plates=[])
        self.shedScores = [WatershedFlow(ws, ws.max_flow_hex()[1]) for ws in self.world.basins.sheds]
        self.shedScores.sort(key=lambda x: x[1], reverse=True)

        watershed_map = np.full(len(terrain.elevations), -1)  # -1 = no watershed
        for score in enumerate(self.shedScores):
            ws_id, watFlow = score
            watershed = watFlow[0]
            for hex_idx in watershed.region.hexes:
                watershed_map[hex_idx] = ws_id

        self.watershed_map = watershed_map

        # mark the map where we can have things
        terrain = self.terrain
        countries = np.zeros(len(terrain.elevations))
        for i in range(len(terrain.elevations)):
            if terrain.elevations[i] < 1:
                countries[i] = -1
            if terrain.elevations[i] > terrain.elevationDelta * (len(terrain.colorLevels)-2):
                countries[i] = -1
        
        if len(self.world.basins.sheds) < 0:
            if debug:
                print("no water")
            self.kingdoms = []
            return

        kingdoms = Kingdom.start(self,self.shedScores[:top_n])
        for i, country in enumerate(kingdoms):

            country.countryName =  GameBoard.completeCountry(country.flag.name,country.flag.countryPrefix)
            for h in country.region:
                countries[h] = country.countryId

        terrain.fields["country"] = countries

        self.kingdoms = kingdoms

    @classmethod
    def completeCountry(cls, name ,place, pattern=None, descriptor=None):
        

        # Possessive patterns
        PATTERNS = [
            "{name}'s {place}",      # Karl's Kingdom
            "{place} of {name}",     # Kingdom of Karl
            "{name} {place}",        # Karl Kingdom
        ]

        if not name:
            return ""
        
        # Choose pattern
        if pattern is not None and 0 <= pattern < len(PATTERNS):
            template = PATTERNS[pattern]
        else:
            template = random.choice(PATTERNS)
        
        return template.format(name=name, place=place)
    
    @staticmethod
    def decode(s: str) -> 'GameBoard':
        """Decode GameBoard from string format."""
        lines = s.strip().split('\n')
        
        world = None
        kingdoms = []
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('+world'):
                world_lines = []
                i += 1
                while not lines[i].startswith('-world'):
                    world_lines.append(lines[i])
                    i += 1
                world = Geology.decode('\n'.join(world_lines))
            
            elif line.startswith('+kingdoms:'):
                i += 1
                while not lines[i].startswith('-kingdoms'):
                    if lines[i].startswith('+kingdom'):
                        kingdom_lines = []
                        i += 1
                        while not lines[i].startswith('-kingdom'):
                            kingdom_lines.append(lines[i])
                            i += 1
                        kingdoms.append(Kingdom.decode('\n'.join(kingdom_lines), world))
                    i += 1
            
            i += 1
        
        # Create GameBoard without triggering __init__
        board = object.__new__(GameBoard)
        board.terrain = world.terrain
        board.world = world
        board.kingdoms = kingdoms
        
        # Reconstruct derived fields
        board.shedScores = [WatershedFlow(ws, ws.max_flow_hex()[1]) for ws in world.basins.sheds]
        board.shedScores.sort(key=lambda x: x[1], reverse=True)
        
        # Rebuild watershed_map
        watershed_map = np.full(len(board.terrain.elevations), -1)
        for ws_id, watFlow in enumerate(board.shedScores):
            watershed = watFlow.watershed
            for hex_idx in watershed.region.hexes:
                watershed_map[hex_idx] = ws_id
        board.watershed_map = watershed_map
        
        return board

In [ ]:
@patch
def encode(self: GameBoard) -> str:
    """Encode GameBoard to string format."""
    lines = []
    
    # We encode world (which contains terrain and basins)
    lines.append("+world")
    lines.append(self.world.encode())
    lines.append("-world")
    
    # Kingdoms
    lines.append(f"+kingdoms:{len(self.kingdoms)}")
    for kingdom in self.kingdoms:
        lines.append("+kingdom")
        lines.append(kingdom.encode())
        lines.append("-kingdom")
    lines.append("-kingdoms")
    
    return '\n'.join(lines)

#| export
I would like to encode and decode GameBoard

In [ ]:
def demoKingdoms():

    sampleMap = TerraDemo().aussie_map()
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleWorld = GameBoard(sampleMap,top_n=7)
    
    return sampleWorld

ourBoard = demoKingdoms()
#print(countries)
for country in ourBoard.kingdoms:
    
    print(f"{country.countryName} ruled by {country.flag.name} has {len(country.settlements)}")

aussieText = ourBoard.encode()

In [ ]:
Watershed.decode??

In [ ]:
demoBoard = GameBoard.decode(aussieText)
for country in demoBoard.kingdoms:
    
    print(f"{country.countryName} ruled by {country.flag.name} has {len(country.settlements)}")

In [ ]:
#| export
@patch
def explore(self: GameBoard, countryId) -> Watershed | None:
    country = [x for x in self.kingdoms if x.countryId == countryId][0]
    capital = country.settlements[0]
    grid = self.world.terrain.hexGrid
    countries = self.terrain.fields["country"]
    
    candidate_watersheds = []
    
    for direction in HexPosition.directions():
        spot = grid.index_to_hexposition(capital.location)
        place = capital.location
        
        # Walk in this direction until we leave the kingdom
        while 0 <= place < len(countries) and countries[place] == countryId:
            spot = spot + direction
            place = grid.hexposition_to_index(spot)
        
        # If we hit unclaimed land, check its watershed
        if 0 <= place < len(countries) and countries[place] == 0:
            ws_id = self.watershed_map[place]
            if ws_id >= 0:
                score = self.shedScores[ws_id]
                candidate_watersheds.append(score)
    
    # Return best watershed by flow, or None
    if candidate_watersheds:
        return max(candidate_watersheds, key=lambda x: x[1])
    return None


In [ ]:
#| export
@patch
def expand_kingdoms(self: GameBoard, max_rounds: int = 100):
    """Expand all kingdoms round-robin style until no more growth possible."""
    countries = self.terrain.fields["country"]
    
    for round_num in range(max_rounds):
        any_growth = False
        
        for kingdom in self.kingdoms:
            score = self.explore(kingdom.countryId)
            
            if score is not None:
                any_growth = True
                # Claim all hexes in this watershed
                for hex_idx in score.watershed.region.hexes:
                    countries[hex_idx] = kingdom.countryId
                    kingdom.region.hexes.add(hex_idx)
                kingdom.captured.append(score)
        
        # Stop if no kingdom could grow
        if not any_growth:
            print(f"Expansion complete after {round_num + 1} rounds")
            break
    
    return self.kingdoms


### Drawing

In [ ]:
#| export
@patch
def find_adjacent_kingdoms(self: Kingdom, all_kingdoms: list['Kingdom']) -> list[int]:
    """Find kingdoms that share a border with this one."""
    adjacent = []
    outside_hexes = self.region.outside(ring=1)
    
    for i, other in enumerate(all_kingdoms):
        if other is self:
            continue
        
        # Check if any of our outside hexes are in their region
        if any(hex_idx in other.region.hexes for hex_idx in outside_hexes.hexes):
            adjacent.append(i)
    
    return adjacent




In [ ]:
#| export
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost




In [ ]:
#| export
@patch
def find_path_dijkstra(self: GameBoard, start_hex: int, end_hex: int) -> list[HexPosition] | None:
    """Find shortest path using Dijkstra's algorithm with elevation costs."""
    #import heapq this is installed in the header

    class BHeapItem:
        def __init__(self, cost, hex_pos):
            self.cost = cost
            self.hex_pos = hex_pos

        def __lt__(self, other):
            return self.cost < other.cost

    grid = self.terrain.hexGrid
    
    # Convert start to HexPosition (using start as origin)
    start_pos = grid.index_to_hexposition(start_hex, start_hex)  # (0,0,0)
    end_pos = grid.index_to_hexposition(end_hex, start_hex)
    
    # Priority queue: (cost, HexPosition)
    queue = [BHeapItem(0, start_pos)]
    costs = {start_pos: 0}
    came_from = {}
    
    while queue:
        item = heapq.heappop(queue)
        current_cost, current = item.cost, item.hex_pos
        
        if current == end_pos:
            # Reconstruct path
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start_pos)
            return list(reversed(path))
        
        # Skip if we've found a better path already
        if current_cost > costs.get(current, float('inf')):
            continue
        
        # Check all 6 neighbors
        for direction in HexPosition.directions():
            neighbor = current + direction
            
            cost = self.movement_cost(current, neighbor, start_hex)
            if cost == float('inf'):
                continue
            
            new_cost = current_cost + cost
            
            if new_cost < costs.get(neighbor, float('inf')):
                costs[neighbor] = new_cost
                came_from[neighbor] = current
                heapq.heappush(queue, BHeapItem(new_cost, neighbor))
    
    return None  # No path found

Any thoughts on where to go next?


In [ ]:
def califorina_place(top_n=5):

    sampleMap = TerraDemo().california_map().downsample_climate(0.25)
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleMap.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(sampleMap,top_n=5)
    sampleWorld.expand_kingdoms(max_rounds=50)
    
    return sampleWorld

cali = califorina_place()

In [ ]:
!cat ../../HexMagic/database.py

So the big question is how to hook all of this up to the database. I do think we want a small meta layer that would be string encodable. but there are details like pieces, settlments, countries that feel they should have their tables. thoughts?

I think this is right. we do need to know which hex is in a kingdom. And I think these tables are good. I do like caching trade_routes since they aren't o N to compute.


So is it better to go down kingdom -> settlement -> piece. or the reverse. the kingdom won't be complete with out the other tables

Yes lets do the piece. should we subclasss GeoStorage or extend it?

Lets go for these classes. GeoStorageDebugger will still work as a temp enviornment with the subclass

In [ ]:
@dataclass
class PieceRecord:
    id: str = ""
    kingdom_id: int = 0
    settlement_id: str = ""  # empty if not in a settlement
    location: int = 0
    owner_id: int = 0
    size: int = 100
    health: int = 100
    max_health: int = 100
    sight: int = 3
    memory: float = 0.8
    movement_range: int = 4
    harvest_goal: str = ""  # store enum value as string
    settle_progress: int = 0
    settle_threshold: int = 3
    world_id: int = 0


In [ ]:
@patch
def save(self: Piece, db: GameStorage = None, world_id: int = 0, 
         settlement_id: str = "", kingdom_id: int = 0):
    """Save this piece to the database as a PieceRecord."""
    db = db or self.db
    if db is None:
        raise ValueError("No database — pass db or set piece.db")
    
    record = PieceRecord(
        id=self.id,
        kingdom_id=kingdom_id or self.owner_id,
        settlement_id=settlement_id,
        location=self.location or -1,
        owner_id=self.owner_id,
        size=self.size,
        health=self.health,
        max_health=self.max_health,
        sight=self.sight,
        memory=self.memory,
        movement_range=self.movement_range,
        harvest_goal=self.harvest_goal.value if self.harvest_goal else "",
        settle_progress=self.settle_progress,
        settle_threshold=self.settle_threshold,
        world_id=world_id,
    )
    db.pieces.upsert(record, pk='id')
    return record


@staticmethod
def from_db(db: GameStorage, piece_id: str) -> 'Piece':
    """Load a Piece from the database by ID."""
    row = db.pieces[piece_id]
    if row is None:
        raise KeyError(f"Piece {piece_id} not found")
    
    r = row if isinstance(row, dict) else dict(row)
    return Piece(
        id=r['id'],
        owner_id=r['owner_id'],
        location=r['location'] if r['location'] >= 0 else None,
        size=r['size'],
        health=r['health'],
        max_health=r['max_health'],
        sight=r['sight'],
        memory=r['memory'],
        movement_range=r['movement_range'],
        harvest_goal=Resources(r['harvest_goal']) if r['harvest_goal'] else None,
        settle_progress=r['settle_progress'],
        settle_threshold=r['settle_threshold'],
    )

Piece.from_db = staticmethod(Piece.from_db) if not hasattr(Piece, 'from_db') else Piece.from_db


## Game Storage

In [ ]:
class GameStorage(GeoStorage):
    def createDB(self):
        print("creating game storage")
        super().createDB()
        self.db.create(PieceRecord, pk='id', if_not_exists=True, transform=True)
        self.pieces = self.db.t.piece_record


In [ ]:
@patch
def pieceOverlay(self:GameStorage,terrain:Terrain,pieces:[Piece])->str:
    """ return a string that can be used to overlay pieces on a map """
    pieceStyle = StyleCSS("piece",fill="red",stroke="black")
    terrain.hexGrid.builder.add_style(pieceStyle)
    retStr = ""
    for piece in pieces:
        coords = terrain.hexGrid.hexes[piece.location].center
        retStr = retStr + f"<circle cx='{coords.x}' cy='{coords.y}' r='10' class='{pieceStyle.name}'></circle>\n"
    return retStr

In [ ]:
@patch
def pieceOverlay(self: GameStorage, terrain: Terrain, world_id: int, 
                 mapper: Callable[[int], int] = None) -> str:
    """Draw pieces from the database onto a terrain map."""
    if mapper is None:
        mapper = lambda idx: idx
    
    pieceStyle = StyleCSS("piece", fill="red", stroke="black")
    terrain.hexGrid.builder.add_style(pieceStyle)
    
    # Query all pieces for this world — grab cursor before fetchall
    cursor = self.db.execute(
        "SELECT * FROM piece_record WHERE world_id = ?", [world_id]
    )
    cols = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    
    if not rows:
        return ""
    
    pieces = [dict(zip(cols, row)) for row in rows]
    
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)
    retStr = ""
    
    for p in pieces:
        loc = p.get('location')
        if loc is None or loc < 0:
            continue
        
        local_idx = mapper(loc)
        if local_idx < 0 or local_idx >= num_hexes:
            continue
        
        coords = grid.hexes[local_idx].center
        retStr += (
            f"<circle cx='{coords.x}' cy='{coords.y}' r='10' "
            f"class='{pieceStyle.name}'></circle>\n"
        )
    
    return retStr


## Building

In [ ]:
SVGBuilder.BUILDERHIDE = False

In [ ]:
# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:

    dbg.server = GameStorage(custom_path=dbg.db_path)
  
    # 1. Load Sydney terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.region.hexes)} hexes")
    #els =  [terrain.elevations[i] for i in target_shed.region.hexes]
    #print(els)
    
    # Convert Watershed hexes to HexRegion
    region = target_shed.region
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    #for i in range(len(terrain.elevations)):
    #    if i not in region:
    #        terrain.elevations[i] = 3000
    #terrain.textElevations()
    print("now detail")

    # result.basins has watersheds computed on the fine (zoomed) grid
    # Score them and pick the best — it corresponds to our target watershed
    fine_scores = [(ws, ws.max_flow_hex()[1]) for ws in result.basins.sheds]
    fine_scores.sort(key=lambda x: x[1], reverse=True)

    # The top watershed in the zoomed region is our target
    best_ws = fine_scores[0][0]
    capital_hex = best_ws.max_flow_hex()[0]

    print(f"Capital hex: {capital_hex}")
    print(f"Watershed has {len(best_ws.region.hexes)} hexes in zoomed grid")
    piece = Piece(location=capital_hex, owner_id=1, current_position=capital_hex)
    piece.db = dbg.server
    piece.save(world_id=cover.ident)
   



    
    # 6. Visualize
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    #capital = ws.max_flow_hex()[0]

    #result.terrain.textElevations()
    terrain = result.terrain
    over = dbg.server.pieceOverlay(result.terrain, world_id=cover.ident)
    
    terrain.hexGrid.builder.adjust("pieces",over)
    print(f"result Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
result.terrain.hexGrid.builder.show()

so I wan

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The zoomed `result.terrain` has its own freshly-computed basins in `result.basins`. Since you zoomed specifically on that watershed's region, the dominant watershed in the result should correspond to the one you targeted. The simplest approach:

```python
# result.basins has watersheds computed on the fine (zoomed) grid
# Score them and pick the best — it corresponds to our target watershed
fine_scores = [(ws, ws.max_flow_hex()[1]) for ws in result.basins.sheds]
fine_scores.sort(key=lambda x: x[1], reverse=True)

# The top watershed in the zoomed region is our target
best_ws = fine_scores[0][0]
capital_hex = best_ws.max_flow_hex()[0]

print(f"Capital hex: {capital_hex}")
print(f"Watershed has {len(best_ws.region.hexes)} hexes in zoomed grid")
```

This works because `cover.zoom_region(region, ...)` builds a finer grid covering that region and then runs `DrainageBasins` on it (that's what `compute_weather=True` triggers). So `result.basins.sheds` are already in the coordinate space of `result.terrain.hexGrid` — no mapping needed.

From there you can drop a piece right on it:

```python
piece = Piece(location=capital_hex, owner_id=1, current_position=capital_hex)
```

If you also want **all the hexes** in the zoomed grid that belong to that watershed (e.g. to create a kingdom region), that's just `best_ws.region` — it's already a `HexRegion` on the fine grid.

One thing to watch: if the zoom splits the original watershed into multiple sub-watersheds (higher resolution reveals ridgelines the coarse grid missed), you might want to grab the top 2-3 and merge their regions. You'll be able to tell by checking if `fine_scores[0][1]` accounts for most of the flow you expected.

So I want to modify pieceOverlay so that it will take the pieces from the database and draw them on the board. we probably need some mapper passed into to map the pieces location in absolute space to the terrain and do a broad filter where get pieces in approximate hexes before checking to see if they are on the map.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good plan. A few design thoughts:

The mapper signature would be something like `world_idx → local_idx` (returning -1 if off-map). You already have examples of this pattern — `CountryDetails.mapper` is a `dict[source, dest]`, and `result.mapper` from `zoom_region` goes fine→coarse.

For the broad filter, since `PieceRecord` has `kingdom_id` and `location`, you could:

1. Query by `world_id` (and optionally `kingdom_id`) to get a rough set
2. Run each through the mapper, keeping only those where `mapper(piece.location) >= 0`
3. Draw only the survivors

One question before we write it: should the mapper be a `dict[int,int]` (like `CountryDetails.mapper`) or a `Callable[[int], int]` (like the zoom mapper)? A callable is more general — you could wrap a dict in a lambda. Want to go with callable?

I think callable is going to make more sense. as we start to build up the database we are going to have to redo CountryDetails (but it is ok now as a reference). The goal is to first be able to draw pieces, then settlments, and finally kingdoms.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Makes sense — callable gives you a clean abstraction that works with both the dict-based `CountryDetails` mapper and the zoom `result.mapper`.

So the new signature would look like:

```python
def pieceOverlay(self:GameStorage, terrain:Terrain, world_id:int, mapper:Callable[[int], int]) -> str:
```

The flow inside would be:

1. Query all `PieceRecord` rows for that `world_id`
2. For each, call `mapper(piece.location)` — skip if result is `-1` or `None`
3. Use the mapped local index to get hex center coords for drawing

Want to take a crack at writing it? The query part is just `self.pieces(where=f"world_id={world_id}")` or similar with fastlite's API.

Can you write it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def pieceOverlay(self: GameStorage, terrain: Terrain, world_id: int, 
                 mapper: Callable[[int], int] = None) -> str:
    """Draw pieces from the database onto a terrain map.
    
    Args:
        terrain: The terrain to draw on
        world_id: Which world's pieces to query
        mapper: Callable mapping absolute hex index → local terrain index.
                Returns -1 if off-map. If None, uses identity (no mapping).
    """
    if mapper is None:
        mapper = lambda idx: idx  # identity — assume pieces are in local coords
    
    pieceStyle = StyleCSS("piece", fill="red", stroke="black")
    terrain.hexGrid.builder.add_style(pieceStyle)
    
    # Query all pieces for this world
    rows = list(self.db.execute(
        "SELECT * FROM piece_record WHERE world_id = ?", [world_id]
    ).fetchall())
    
    if not rows:
        return ""
    
    cols = [d[0] for d in self.db.execute(
        "SELECT * FROM piece_record LIMIT 0"
    ).description]
    pieces = [dict(zip(cols, row)) for row in rows]
    
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)
    retStr = ""
    
    for p in pieces:
        loc = p.get('location')
        if loc is None or loc < 0:
            continue
        
        # Map from absolute world space to local terrain space
        local_idx = mapper(loc)
        if local_idx < 0 or local_idx >= num_hexes:
            continue
        
        coords = grid.hexes[local_idx].center
        retStr += (
            f"<circle cx='{coords.x}' cy='{coords.y}' r='10' "
            f"class='{pieceStyle.name}'></circle>\n"
        )
    
    return retStr
```

And you'd use it like:

```python
# With zoom_region result (fine→coarse mapper needs inverting)
# Build a world_idx → local_idx mapper from the zoom result
inv_mapper = {}
for fine_idx in range(len(result.terrain.hexGrid.hexes)):
    coarse_idx = result.mapper(fine_idx)
    if coarse_idx >= 0 and coarse_idx not in inv_mapper:
        inv_mapper[coarse_idx] = fine_idx

over = dbg.server.pieceOverlay(
    result.terrain, 
    world_id=cover.ident,
    mapper=lambda idx: inv_mapper.get(idx, -1)
)
result.terrain.hexGrid.builder.adjust("pieces", over)
```

A few things to note:

- The inverse mapper picks the *first* fine hex it finds for each coarse hex. For pieces this is fine since you just need a point to draw on, but if you want centered placement you could average the fine hex coords instead.
- The query grabs *all* pieces for the world — once you have lots of pieces, you'll want a bounding-box pre-filter. That could be a `kingdom_id` filter or a spatial query on `location` range. Easy to add later.
- The same pattern will extend nicely to settlements (different style, maybe a square or star instead of circle) and kingdoms (fill the region hexes with the flag color).

In order to make the demo work I need to have so that pieces can save themselves

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def save(self: Piece, db: GameStorage = None, world_id: int = 0, 
         settlement_id: str = "", kingdom_id: int = 0):
    """Save this piece to the database as a PieceRecord."""
    db = db or self.db
    if db is None:
        raise ValueError("No database — pass db or set piece.db")
    
    record = PieceRecord(
        id=self.id,
        kingdom_id=kingdom_id or self.owner_id,
        settlement_id=settlement_id,
        location=self.location or -1,
        owner_id=self.owner_id,
        size=self.size,
        health=self.health,
        max_health=self.max_health,
        sight=self.sight,
        memory=self.memory,
        movement_range=self.movement_range,
        harvest_goal=self.harvest_goal.value if self.harvest_goal else "",
        settle_progress=self.settle_progress,
        settle_threshold=self.settle_threshold,
        world_id=world_id,
    )
    db.pieces.upsert(record, pk='id')
    return record
```

Then in your demo it's just:

```python
piece = Piece(location=capital_hex, owner_id=1, current_position=capital_hex)
piece.db = dbg.server
piece.save(world_id=cover.ident)
```

You could also add a `load` for symmetry:

```python
@staticmethod
def from_db(db: GameStorage, piece_id: str) -> 'Piece':
    """Load a Piece from the database by ID."""
    row = db.pieces[piece_id]
    if row is None:
        raise KeyError(f"Piece {piece_id} not found")
    
    r = row if isinstance(row, dict) else dict(row)
    return Piece(
        id=r['id'],
        owner_id=r['owner_id'],
        location=r['location'] if r['location'] >= 0 else None,
        size=r['size'],
        health=r['health'],
        max_health=r['max_health'],
        sight=r['sight'],
        memory=r['memory'],
        movement_range=r['movement_range'],
        harvest_goal=Resources(r['harvest_goal']) if r['harvest_goal'] else None,
        settle_progress=r['settle_progress'],
        settle_threshold=r['settle_threshold'],
    )

Piece.from_db = staticmethod(Piece.from_db) if not hasattr(Piece, 'from_db') else Piece.from_db
```

The `upsert` with `pk='id'` means you can call `piece.save()` repeatedly — it'll insert the first time and update after that.

Now can over be made from the new pieceOverlay?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Since `capital_hex` already came from the zoomed grid (`result.basins`), the piece location is in the local terrain's coordinate space. So you just need the identity mapper (which is the default):

```python
over = dbg.server.pieceOverlay(result.terrain, world_id=cover.ident)
terrain.hexGrid.builder.adjust("pieces", over)
```

No mapper needed here — the piece was placed at `capital_hex` which is already a fine-grid index. The mapper becomes important when pieces are stored in *coarse* (world) coordinates and you need to project them onto a zoomed view. That's the case I showed with the `inv_mapper` earlier.

So in your demo, just replace:

```python
#over = dbg.server.pieceOverlay(terrain,[piece])
```

with:

```python
over = dbg.server.pieceOverlay(result.terrain, world_id=cover.ident)
```

Something went wrong

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The error is `ExecutionCompleteError: Can't get description for statements that have completed execution` — the second query (`SELECT * ... LIMIT 0`) to grab column names doesn't work with fastlite's cursor behavior. The fix is to get the column names from the *first* query's cursor instead of running a separate one:

```python
@patch
def pieceOverlay(self: GameStorage, terrain: Terrain, world_id: int, 
                 mapper: Callable[[int], int] = None) -> str:
    """Draw pieces from the database onto a terrain map."""
    if mapper is None:
        mapper = lambda idx: idx
    
    pieceStyle = StyleCSS("piece", fill="red", stroke="black")
    terrain.hexGrid.builder.add_style(pieceStyle)
    
    # Query all pieces for this world — grab cursor before fetchall
    cursor = self.db.execute(
        "SELECT * FROM piece_record WHERE world_id = ?", [world_id]
    )
    cols = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    
    if not rows:
        return ""
    
    pieces = [dict(zip(cols, row)) for row in rows]
    
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)
    retStr = ""
    
    for p in pieces:
        loc = p.get('location')
        if loc is None or loc < 0:
            continue
        
        local_idx = mapper(loc)
        if local_idx < 0 or local_idx >= num_hexes:
            continue
        
        coords = grid.hexes[local_idx].center
        retStr += (
            f"<circle cx='{coords.x}' cy='{coords.y}' r='10' "
            f"class='{pieceStyle.name}'></circle>\n"
        )
    
    return retStr
```

The key change: grab `.description` from the cursor *before* calling `.fetchall()`, not from a second query.